In [ ]:
!pip install scikit-learn nltk

In [39]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# Scarica le risorse necessarie per NLTK (solo la prima volta)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

class DyCSCOMAnalyzer:
    def __init__(self):
        self.stemmer = PorterStemmer()
        # Carica le stop words in inglese
        self.stop_words = set(stopwords.words('english'))
        # NUOVO: Dizionario per mappare la radice (stem) alla parola originale
        # Es: {'secur': 'security', 'manag': 'management'}
        self.stem_to_original = {}

    def preprocess(self, text):
        """
        Esegue la pulizia e lo stemming, salvando la mappatura inversa per la visualizzazione.
        """
        if not text:
            return ""

        # 1. Rimozione HTML tags
        text = re.sub(r'<[^>]+>', '', text)

        # 2. Rimozione caratteri non alfabetici e conversione in minuscolo
        text = re.sub(r'[^a-zA-Z\s]', '', text).lower()

        # 3. Tokenizzazione
        words = text.split()

        stemmed_tokens = []
        for word in words:
            if word not in self.stop_words and len(word) > 2: # Ignoriamo parole troppo brevi
                # Applicazione Porter Stemmer
                stem = self.stemmer.stem(word)
                stemmed_tokens.append(stem)
                
                # NUOVO: Salviamo la parola originale associata a questo stem
                # Se lo stem è nuovo o se abbiamo trovato una parola originale più corta/rappresentativa, aggiorniamo
                if stem not in self.stem_to_original:
                    self.stem_to_original[stem] = word
                else:
                    # Euristica: preferiamo mantenere la variante più corta o frequente se necessario
                    # Qui teniamo semplicemente la prima o la più corta per pulizia
                    if len(word) < len(self.stem_to_original[stem]):
                        self.stem_to_original[stem] = word

        return " ".join(stemmed_tokens)

    def extract_word_importance(self, syllabus_text):
        """
        Applica TF-IDF e NMF per calcolare l'importanza delle parole (stems).
        """
        
        # Passo 1: Pre-processing
        clean_text = self.preprocess(syllabus_text)
        
        if not clean_text:
            return {}

        # Passo 2: TF-IDF Vectorization 
        vectorizer = TfidfVectorizer(use_idf=True, norm='l2', smooth_idf=True)
        
        # Trasformiamo il testo in una matrice sparsa
        tfidf_matrix = vectorizer.fit_transform([clean_text])
        feature_names = vectorizer.get_feature_names_out()

        # Passo 3: Non-Negative Matrix Factorization (NMF) 
        nmf_model = NMF(
            n_components=1, 
            random_state=42, 
            init='random',  # Fix per stabilità su testi singoli
            max_iter=5000, 
            tol=1e-3,       
            solver='mu'     
        )
        nmf_model.fit(tfidf_matrix)        
        
        word_weights = nmf_model.components_[0]

        # Passo 4: Creazione del dizionario Key-Value (Stem -> Peso)
        importance_dict = {
            word: weight 
            for word, weight in zip(feature_names, word_weights) 
            if weight > 0 
        }

        # Ordinamento per importanza decrescente
        sorted_importance = dict(sorted(importance_dict.items(), key=lambda item: item[1], reverse=True))
        
        return sorted_importance

def generate_ecsf_keywords(raw_data_dict):
    analyzer = DyCSCOMAnalyzer()
    ecsf_keywords_table = {}
    
    # Dizionario globale per debug (opzionale)
    all_weighted_keywords = {} 

    for role_name, sections in raw_data_dict.items():
        # 1. AGGREGAZIONE DEL TESTO
        full_description = (
            f"{sections.get('summary', '')} "
            f"{sections.get('mission', '')} "
            f"{sections.get('tasks', '')} "
            f"{sections.get('skills', '')} "
            f"{sections.get('knowledge', '')}"
        )
        
        # 2. ESTRAZIONE (Preprocessing -> TF-IDF -> NMF)
        # weighted_keywords contiene: {'secur': 0.5, 'manag': 0.4 ...}
        weighted_keywords = analyzer.extract_word_importance(full_description)
        all_weighted_keywords[role_name] = weighted_keywords # Salviamo per debug

        # 3. FILTRAGGIO E CONVERSIONE IN PAROLE REALI
        # Prendiamo le top 15 stems
        top_stems = list(weighted_keywords.keys())[:15]
        
        # Convertiamo gli stem in parole originali usando il dizionario dell'analyzer
        top_words_real = []
        for stem in top_stems:
            # Recupera la parola originale, se non c'è (caso raro) usa lo stem
            real_word = analyzer.stem_to_original.get(stem, stem)
            top_words_real.append(real_word)

        ecsf_keywords_table[role_name] = top_words_real
        
    return weighted_keywords,ecsf_keywords_table

In [49]:

# Sillabus NS
import json


syllabus_sample = """
The main security properties of an information system will be introduced and discussed in detail. The approaches for improving security at the various levels of the network protocol stack will then be presented and analyzed. The course adopts an “offensive security” approach. The concepts related to the so-called vulnerability–threat–attack chain will be illustrated. The techniques for preparing a cyber attack, such as footprinting, scanning, and enumeration, will be described. Finally, the final phase of an attack, known as exploitation, will be presented. Topics such as firewalling, intrusion detection, malware analysis, and protection against Distributed Denial of Service (DDoS) attacks will be introduced. Finally, the main techniques of so-called ethical hacking will be presented and analyzed in detail.

Syllabus:

Network security: principles and architecture
Functional security requirements
Threats, attacks, countermeasures

Wireless network security

Network-layer security
The IPsec protocol suite

Transport-layer security
Secure Socket Layer (SSL), Transport Layer Security (TLS), Secure Shell (SSH)

Application-layer security
Email
Web
HTTPS
Security architecture of WebRTC (Web Real-Time Communications)

Cloud computing and security (overview)

Introduction to malware
Taxonomy
Viruses, worms, Trojans, rootkits, etc.
Advanced Persistent Threats (APTs)
Countermeasures

Denial of Service (DoS) and Distributed Denial of Service (DDoS) attacks

Intrusion Detection Systems (IDS)
Host-based, network-based, and hybrid techniques

Firewalls and Intrusion Prevention Systems (IPS)

Hacking in IP networks
Preliminary phases of an attack
Footprinting, scanning, enumeration

Targeted attack techniques
Against end systems and servers
Against infrastructure
VoIP (Voice over IP) networks
Wireless networks
Hardware systems
Against data and applications
Web
Mobile devices
Databases
"""

# Istanzia l'analizzatore
analyzer = DyCSCOMAnalyzer()

# Esegui l'estrazione
result_dictionary = analyzer.extract_word_importance(syllabus_sample)

# Stampa i risultati
print("Dizionario Pesi (Stemmed Words -> Importance Score):")
for word, score in list(result_dictionary.items())[:20]: # Stampa i primi 10
    print(f"Key: '{word}', Value: {score:.4f}")

print(json.dumps(result_dictionary, indent=2))
with open("DYCSCOM_weights.json", "w") as f:
    json.dump(result_dictionary, f, indent=2)

Dizionario Pesi (Stemmed Words -> Importance Score):
Key: 'secur', Value: 2.5980
Key: 'attack', Value: 1.2990
Key: 'network', Value: 1.1134
Key: 'system', Value: 0.9279
Key: 'techniqu', Value: 0.7423
Key: 'denial', Value: 0.5567
Key: 'final', Value: 0.5567
Key: 'intrus', Value: 0.5567
Key: 'present', Value: 0.5567
Key: 'servic', Value: 0.5567
Key: 'web', Value: 0.5567
Key: 'analyz', Value: 0.3711
Key: 'approach', Value: 0.3711
Key: 'architectur', Value: 0.3711
Key: 'countermeasur', Value: 0.3711
Key: 'ddo', Value: 0.3711
Key: 'detail', Value: 0.3711
Key: 'detect', Value: 0.3711
Key: 'distribut', Value: 0.3711
Key: 'enumer', Value: 0.3711
{
  "secur": 2.5980011673317067,
  "attack": 1.2990005836658534,
  "network": 1.1134290717135884,
  "system": 0.9278575597613239,
  "techniqu": 0.7422860478090592,
  "denial": 0.5567145358567942,
  "final": 0.5567145358567942,
  "intrus": 0.5567145358567942,
  "present": 0.5567145358567942,
  "servic": 0.5567145358567942,
  "web": 0.5567145358567942,
 

In [40]:
ecsf_raw_data = {
    "Chief Information Security Officer": {
        "summary": "Manages an organisation's cybersecurity strategy and its implementation to ensure that digital systems, services and assets are adequately secure and protected.",
        "mission": "Defines, maintains and communicates the cybersecurity vision, strategy, policies and procedures. Manages the implementation of the cybersecurity policy across the organisation. Assures information exchange with external authorities and professional bodies.",
        "tasks": "Define, implement, communicate and maintain cybersecurity goals, requirements, strategies, policies, aligned with the business strategy. Prepare and present cybersecurity vision, strategies and policies for approval by the senior management. Supervise the application and improvement of the Information Security Management System ISMS. Educate senior management about cybersecurity risks. Develop cybersecurity plans. Negotiate the cybersecurity budget. Manage continuous capacity building.",
        "skills": "Assess and enhance an organisation's cybersecurity posture. Analyse and implement cybersecurity policies, certifications, standards. Manage cybersecurity resources. Develop, champion and lead the execution of a cybersecurity strategy. Influence an organisation's cybersecurity culture. Design, apply, monitor and review ISMS. Communicate, coordinate and cooperate with internal and external stakeholders.",
        "knowledge": "Cybersecurity policies. Cybersecurity standards, methodologies and frameworks. Cybersecurity recommendations and best practices. Cybersecurity related laws, regulations and legislations. Cybersecurity-related certifications. Ethical cybersecurity organisation requirements. Risk management standards."
    },
    "Cyber Incident Responder": {
        "summary": "Monitor the organisation's cybersecurity state, handle incidents during cyber-attacks and assure the continued operations of ICT systems.",
        "mission": "Monitors and assesses systems' cybersecurity state. Analyses, evaluates and mitigates the impact of cybersecurity incidents. Identifies cyber incidents root causes and malicious actors. Restores systems and processes functionalities to an operational state, collecting evidences and documenting actions taken.",
        "tasks": "Contribute to the development, maintenance and assessment of the Incident Response Plan. Develop, implement and assess procedures related to incident handling. Identify, analyse, mitigate and communicate cybersecurity incidents. Assess and manage technical vulnerabilities. Measure cybersecurity incidents detection and response effectiveness. Cooperate with SOCs and CSIRTs.",
        "skills": "Practice all technical, functional and operational aspects of cybersecurity incident handling. Collect, analyse and correlate cyber threat information. Work on operating systems, servers, clouds and relevant infrastructures. Work under pressure. Manage and analyse log files.",
        "knowledge": "Incident handling standards, methodologies and frameworks. Incident handling recommendations and best practices. Incident handling tools. Operating systems security. Computer networks security. Cyber threats. Cybersecurity attack procedures. Computer systems vulnerabilities. CSIRTs operation."
    },
    "Cyber Legal, Policy & Compliance Officer": {
        "summary": "Manages compliance with cybersecurity-related standards, legal and regulatory frameworks based on the organisation's strategy and legal requirements.",
        "mission": "Oversees and assures compliance with cybersecurity- and data-related legal, regulatory frameworks and policies. Contributes to the organisation's data protection related actions. Provides legal advice in the development of the organisation's cybersecurity governance processes.",
        "tasks": "Ensure compliance with and provide legal advice on data privacy and data protection standards. Identify and document compliance gaps. Conduct privacy impact assessments. Enforce and advocate organisation's data privacy program. Act as a key contact point to handle queries regarding data processing. Manage legal aspects of information security responsibilities.",
        "skills": "Comprehensive understanding of the business strategy, models and products. Carry out working-life practices of the data protection and privacy issues. Lead the development of appropriate cybersecurity and privacy policies. Conduct, monitor and review privacy impact assessments. Understand, practice and adhere to ethical requirements.",
        "knowledge": "Cybersecurity related laws, regulations and legislations. Cybersecurity standards, methodologies and frameworks. Cybersecurity policies. Legal, regulatory and legislative compliance requirements. Privacy impact assessment standards."
    },
    "Cyber Threat Intelligence Specialist": {
        "summary": "Collect, process, analyse data and information to produce actionable intelligence reports and disseminate them to target stakeholders.",
        "mission": "Manages cyber threat intelligence life cycle including cyber threat information collection, analysis and production of actionable intelligence. Identifies and monitors the Tactics, Techniques and Procedures TTPs used by cyber threat actors and their trends.",
        "tasks": "Develop, implement and manage the organisation's cyber threat intelligence strategy. Translate business requirements into Intelligence Requirements. Identify and assess cyber threat actors. Identify, monitor and assess the TTPs used by cyber threat actors. Produce actionable reports based on threat intelligence data. Leverage intelligence data to support threat modelling.",
        "skills": "Collect, analyse and correlate cyber threat information originating from multiple sources. Identify threat actors TTPs and campaigns. Automate threat intelligence management procedures. Conduct technical analysis and reporting. Model threats, actors and TTPs. Use and apply CTI platforms and tools.",
        "knowledge": "Operating systems security. Computer networks security. Cybersecurity controls and solutions. Computer programming. CTI sharing standards. Cyber threats. Cyber threat actors. Advanced and persistent cyber threats APT. Threat actors Tactics, Techniques and Procedures TTPs."
    },
    "Cybersecurity Architect": {
        "summary": "Plans and designs security-by-design solutions (infrastructures, systems, assets, software, hardware and services) and cybersecurity controls.",
        "mission": "Designs solutions based on security-by-design and privacy-by-design principles. Creates and continuously improves architectural models. Coordinate secure development, integration and maintenance of cybersecurity components.",
        "tasks": "Design and propose a secure architecture. Develop organisation's cybersecurity architecture to address security and privacy requirements. Produce architectural documentation and specifications. Establish a secure environment during the development lifecycle. Assure the security of the solution architectures through security reviews.",
        "skills": "Conduct user and business security requirements analysis. Draw cybersecurity architectural and functional specifications. Decompose and analyse systems to develop security and privacy requirements. Design systems based on security and privacy by design. Build resilience against points of failure.",
        "knowledge": "Cybersecurity recommendations and best practices. Cybersecurity standards. Secure development lifecycle. Security architecture reference models. Cybersecurity controls and solutions. Privacy-Enhancing Technologies PET. Privacy-by-design standards."
    },
    "Cybersecurity Auditor": {
        "summary": "Perform cybersecurity audits on the organisation's ecosystem. Ensuring compliance with statutory, regulatory, policy information, security requirements, industry standards and best practices.",
        "mission": "Conducts independent reviews to assess the effectiveness of processes and controls and the overall compliance. Evaluates, tests and verifies cybersecurity-related products, functions and policies ensuring compliance.",
        "tasks": "Develop the organisation's auditing policy, procedures, standards. Define audit scope, objectives and criteria. Develop an audit plan. Audit compliance with cybersecurity-related applicable laws and regulations. Execute the audit plan and collect evidence. Monitor risk remediation activities.",
        "skills": "Organise and work in a systematic and deterministic way based on evidence. Follow and practice auditing frameworks. Apply auditing tools and techniques. Decompose and analyse systems to identify weaknesses. Audit with integrity, being impartial and independent.",
        "knowledge": "Cybersecurity controls and solutions. Legal, regulatory and legislative compliance requirements. Monitoring, testing and evaluating cybersecurity controls effectiveness. Conformity assessment standards. Auditing standards, methodologies and frameworks."
    },
    "Cybersecurity Educator": {
        "summary": "Improves cybersecurity knowledge, skills and competencies of humans.",
        "mission": "Designs, develops and conducts awareness, training and educational programmes in cybersecurity. Uses appropriate teaching and training methods to communicate and enhance the cybersecurity culture. Promotes the importance of cybersecurity.",
        "tasks": "Develop, update and deliver cybersecurity curricula and educational material. Organise, design and deliver cybersecurity awareness-raising activities. Monitor, evaluate and report training effectiveness. Design, develop and deliver cybersecurity simulations or cyber range environments.",
        "skills": "Identify needs in cybersecurity awareness, training and education. Design, develop and deliver learning programmes. Develop cybersecurity exercises including simulations. Provide training towards cybersecurity certifications. Identify and select appropriate pedagogical approaches.",
        "knowledge": "Pedagogical standards, methodologies and frameworks. Cybersecurity awareness, education and training programme development. Cybersecurity education and training standards. Cybersecurity recommendations and best practices."
    },
    "Cybersecurity Implementer": {
        "summary": "Develop, deploy and operate cybersecurity solutions (systems, assets, software, controls and services) on infrastructures and products.",
        "mission": "Provides cybersecurity-related technical development, integration, testing, implementation, operation, maintenance, monitoring and support of cybersecurity solutions. Ensures adherence to specifications and conformance requirements.",
        "tasks": "Develop, implement, maintain, upgrade, test cybersecurity products. Provide cybersecurity-related support to users. Integrate cybersecurity solutions. Securely configure systems, services and products. Implement, apply and manage patches to products to address technical vulnerabilities.",
        "skills": "Integrate cybersecurity solutions to the organisation's infrastructure. Configure solutions according to the organisation's security policy. Assess the security and performance of solutions. Develop code, scripts and programmes. Identify and solve cybersecurity-related issues.",
        "knowledge": "Secure development lifecycle. Computer programming. Operating systems security. Computer networks security. Cybersecurity controls and solutions. Offensive and defensive security practices. Secure coding recommendations."
    },
    "Cybersecurity Researcher": {
        "summary": "Research the cybersecurity domain and incorporate results in cybersecurity solutions.",
        "mission": "Conducts fundamental/basic and applied research and facilitates innovation in the cybersecurity domain. Analyses trends and scientific findings in cybersecurity.",
        "tasks": "Analyse and assess cybersecurity technologies. Conduct research, innovation and development work. Advance the current state-of-the-art. Conduct experiments and develop a proof of concept. Publish and present scientific works and research and development results.",
        "skills": "Generate new ideas and transfer theory into practice. Decompose and analyse systems to identify weaknesses. Monitor new advancements in cybersecurity-related technologies. Identify and solve cybersecurity-related issues.",
        "knowledge": "Cybersecurity-related research, development and innovation RDI. Cybersecurity standards, methodologies and frameworks. Legal requirements on releasing cybersecurity related technologies. Multidiscipline aspect of cybersecurity."
    },
    "Cybersecurity Risk Manager": {
        "summary": "Manage the organisation's cybersecurity-related risks aligned to the organisation's strategy. Develop, maintain and communicate the risk management processes and reports.",
        "mission": "Continuously manages (identifies, analyses, assesses, estimates, mitigates) the cybersecurity-related risks of ICT infrastructures. Establishes a risk management strategy and ensures that risks remain at an acceptable level.",
        "tasks": "Develop an organisation's cybersecurity risk management strategy. Manage an inventory of organisation's assets. Identify and assess cybersecurity-related threats and vulnerabilities. Assess cybersecurity risks and propose risk treatment options. Monitor effectiveness of cybersecurity controls.",
        "skills": "Implement cybersecurity risk management frameworks. Analyse and consolidate organisation's quality and risk management practices. Enable business assets owners to make risk-informed decisions. Build a cybersecurity risk-aware environment.",
        "knowledge": "Risk management standards, methodologies and frameworks. Risk management tools. Cyber threats. Computer systems vulnerabilities. Cybersecurity controls and solutions. Cybersecurity risks. Monitoring, testing and evaluating cybersecurity controls effectiveness."
    },
    "Digital Forensics Investigator": {
        "summary": "Ensure the cybercriminal investigation reveals all digital evidence to prove the malicious activity.",
        "mission": "Connects artefacts to natural persons, captures, recovers, identifies and preserves data. Provides analysis, reconstruction and interpretation of the digital evidence based on a qualitative opinion.",
        "tasks": "Develop digital forensics investigation policy. Identify, recover, extract, document and analyse digital evidence. Preserve and protect digital evidence. Inspect environments for evidence of unauthorised actions. Document, report and present digital forensic analysis findings.",
        "skills": "Work ethically and independently. Collect information while preserving its integrity. Identify, analyse and correlate cybersecurity events. Explain and present digital evidence. Develop and communicate detailed investigation reports.",
        "knowledge": "Digital forensics recommendations and best practices. Digital forensics standards. Digital forensics analysis procedures. Criminal investigation procedures. Malware analysis tools. Cyber threats. Computer systems vulnerabilities."
    },
    "Penetration Tester": {
        "summary": "Assess the effectiveness of security controls, reveals and utilise cybersecurity vulnerabilities, assessing their criticality if exploited by threat actors.",
        "mission": "Plans, designs, implements and executes penetration testing activities and attack scenarios to evaluate the effectiveness of deployed or planned security measures. Identifies vulnerabilities or failures on technical and organisational controls.",
        "tasks": "Identify, analyse and assess technical and organisational cybersecurity vulnerabilities. Identify attack vectors, uncover and demonstrate exploitation. Select and develop appropriate penetration testing techniques. Organise test plans and procedures. Deploy penetration testing tools.",
        "skills": "Develop codes, scripts and programmes. Perform social engineering. Identify and exploit vulnerabilities. Conduct ethical hacking. Think creatively and outside the box. Use penetration testing tools effectively. Review codes assess their security.",
        "knowledge": "Cybersecurity attack procedures. IT and OT appliances. Offensive and defensive security procedures. Operating systems security. Computer networks security. Penetration testing procedures. Penetration testing standards and tools. Computer programming."
    }
}
ecsf_dictionary,ecsf_keywords_table = generate_ecsf_keywords(ecsf_raw_data)

for role, keywords in ecsf_keywords_table.items():
    print(f"Role: {role}, Top Keywords: {keywords}")

Role: Chief Information Security Officer, Top Keywords: ['cybersecurity', 'manage', 'policy', 'strategy', 'organisation', 'implement', 'communicate', 'standards', 'certifications', 'define', 'develop', 'external', 'information', 'isms', 'maintain']
Role: Cyber Incident Responder, Top Keywords: ['incident', 'cybersecurity', 'handle', 'operating', 'system', 'analyse', 'assess', 'cyber', 'state', 'collect', 'computer', 'csirts', 'develop', 'functional', 'identify']
Role: Cyber Legal, Policy & Compliance Officer, Top Keywords: ['legal', 'privacy', 'cybersecurity', 'data', 'compliance', 'organisation', 'standards', 'assess', 'frameworks', 'impact', 'policy', 'protected', 'regulatory', 'requirements', 'advice']
Role: Cyber Threat Intelligence Specialist, Top Keywords: ['threat', 'cyber', 'intelligence', 'actors', 'ttps', 'identify', 'actions', 'collect', 'data', 'information', 'manage', 'procedures', 'reports', 'use', 'analyse']
Role: Cybersecurity Architect, Top Keywords: ['secure', 'archit

In [44]:
def calculate_coverage_percentage(syllabus_weights, ecsf_stems_table):
    """
    Calcola la Coverage in PERCENTUALE (%).
    
    Formula: (Score Calcolato / Score Massimo Teorico) * 100
    Dove Score Massimo Teorico = Somma dei KIF (assumendo match perfetto = 1.0)
    """
    KIF_ARRAY = [2.0, 1.7, 1.5, 1.3, 1.2, 1.15, 1.1, 1.05, 1.0, 1.0]
    coverage_results = {}

    for role, role_stems in ecsf_stems_table.items():
        current_score = 0.0
        max_theoretical_score = 0.0
        
        # Consideriamo al massimo tante keyword quanti sono i KIF disponibili
        limit = min(len(role_stems), len(KIF_ARRAY))
        
        for i in range(limit):
            stem = role_stems[i]
            kif_value = KIF_ARRAY[i]
            
            # 1. Calcolo Score Attuale
            nmf_weight = syllabus_weights.get(stem, 0.0)
            current_score += nmf_weight * kif_value
            
            # 2. Calcolo Score Massimo (Se il syllabus avesse questa parola con peso max 1.0)
            # Questo serve per il denominatore della percentuale
            max_theoretical_score += 1.0 * kif_value
            
        # Evitiamo divisioni per zero
        if max_theoretical_score > 0:
            percentage = (current_score / max_theoretical_score) * 100.0
        else:
            percentage = 0.0
            
        coverage_results[role] = percentage

    return coverage_results

In [45]:
cybersecurity_roles = {
    "Chief Information Security Officer": "cybersecurity management senior related organisation risks policies organisations develop ensure",
    "Cyber Incident Responder": "incident handling procedures cybersecurity response develop analysis results related actions",
    "Cyber Legal Policy & Compliance Officer": "data protection privacy compliance cybersecurity organisations legal ensure standards communicate",
    "Cyber Threat Intelligence Specialist": "threat intelligence cyber actors procedures threats identify ttps information data",
    "Cybersecurity Architect": "cybersecurity architecture security organisations related design requirements solutions evaluate privacy",
    "Cybersecurity Auditor": "standards auditing frameworks methodologies cybersecurity audit develop conformity procedures related",
    "Cybersecurity Educator": "cybersecurity training related awareness education develop deliver data protection methodologies",
    "Cybersecurity Implementer": "cybersecurity related solutions controls integrate performance security implement organisations products",
    "Cybersecurity Researcher": "cybersecurity related solutions development technologies identify research innovation topics ideas",
    "Cybersecurity Risk Manager": "cybersecurity risk controls management related organisations risks effectiveness practices strategy",
    "Digital Forensics Investigator": "digital forensics evidence procedures analysis present investigation document testing develop",
    "Penetration Tester": "testing penetration tools procedures test standards develop stakeholders report analysis"
}
#usiamo la stessa tabella di keywords ECSF del paper visto che la nostra è leggermente differente


from nltk.stem import PorterStemmer

# 1. Inizializza lo Stemmer
stemmer = PorterStemmer()


# 3. Funzione per applicare lo stemming
def stem_roles_dictionary(roles_dict):
    stemmed_dict = {}
    
    for role, keywords_str in roles_dict.items():
        # A. Divide la stringa in parole singole
        words = keywords_str.split()
        
        # B. Applica lo stemmer a ogni parola
        # Nota: assumiamo che siano già pulite/lowercase come da tuo input
        stems_list = [stemmer.stem(word) for word in words]
        
        # C. Salva la lista di stems nel nuovo dizionario
        stemmed_dict[role] = stems_list
        
    return stemmed_dict

# --- ESECUZIONE ---
ecsf_stems_table = stem_roles_dictionary(cybersecurity_roles)


for role, keywords in ecsf_stems_table.items():
    print(f"Role: {role}, Stemmed Keywords: {keywords}")

Role: Chief Information Security Officer, Stemmed Keywords: ['cybersecur', 'manag', 'senior', 'relat', 'organis', 'risk', 'polici', 'organis', 'develop', 'ensur']
Role: Cyber Incident Responder, Stemmed Keywords: ['incid', 'handl', 'procedur', 'cybersecur', 'respons', 'develop', 'analysi', 'result', 'relat', 'action']
Role: Cyber Legal Policy & Compliance Officer, Stemmed Keywords: ['data', 'protect', 'privaci', 'complianc', 'cybersecur', 'organis', 'legal', 'ensur', 'standard', 'commun']
Role: Cyber Threat Intelligence Specialist, Stemmed Keywords: ['threat', 'intellig', 'cyber', 'actor', 'procedur', 'threat', 'identifi', 'ttp', 'inform', 'data']
Role: Cybersecurity Architect, Stemmed Keywords: ['cybersecur', 'architectur', 'secur', 'organis', 'relat', 'design', 'requir', 'solut', 'evalu', 'privaci']
Role: Cybersecurity Auditor, Stemmed Keywords: ['standard', 'audit', 'framework', 'methodolog', 'cybersecur', 'audit', 'develop', 'conform', 'procedur', 'relat']
Role: Cybersecurity Educa

In [50]:
final_scores = calculate_coverage_percentage(result_dictionary, ecsf_stems_table)

# D. Visualizzazione (Convertiamo gli stems in parole reali QUI)
print(f"\n{'RUOLO':<40} | {'SCORE (%)':<10} | {'TOP KEYWORDS MATCHED (Real Words)'}")
print("-" * 100)

# Ordiniamo per punteggio
sorted_roles = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)


for role, score in sorted_roles:
    # Recuperiamo le keyword del ruolo che hanno contribuito (quelle presenti nel syllabus)
    role_stems = ecsf_stems_table[role][:10] # Top 10 usate per il calcolo
    
    matched_words = []
    
    for stem in role_stems:
        # Recupera la parola originale, se non c'è (caso raro) usa lo stem
        real_word = analyzer.stem_to_original.get(stem, stem)
        
        # Verifica se questa parola/stem ha un peso > 0 nel syllabus
        if stem in result_dictionary:
            matched_words.append(real_word)
    
    # Stampa il risultato formattato
    print(f"{role:<40} | {score:<10.4f} | {', '.join(matched_words)}")


RUOLO                                    | SCORE (%)  | TOP KEYWORDS MATCHED (Real Words)
----------------------------------------------------------------------------------------------------
Cybersecurity Architect                  | 38.1135    | architecture, secure, related, requirements
Cybersecurity Implementer                | 24.4098    | related, secure
Cyber Threat Intelligence Specialist     | 13.9892    | threats, cyber, threats, information, data
Cyber Legal Policy & Compliance Officer  | 6.7091     | data, protection, communications
Digital Forensics Investigator           | 6.6378     | analysis, presented
Cybersecurity Educator                   | 5.0675     | related, data, protection
Cybersecurity Researcher                 | 3.8542     | related, topics
Cyber Incident Responder                 | 2.9977     | analysis, related
Chief Information Security Officer       | 1.8557     | related
Cybersecurity Risk Manager               | 1.7130     | related
Cybersecurity Au

"Gap Semantico" (Semantic Gap).

(Il problema della "Bag of Words") 
L'algoritmo che stiamo usando (TF-IDF + NMF) è un approccio lessicale, non semantico. Questo significa che confronta le parole (stringhe di testo), non i significati.Guarda cosa sta succedendo "sotto il cofano":
Il Dizionario ENISA (Reference) vuole queste parole:
['test', 'penetr', 'tool', 'procedur', 'test', 'standard', 'develop', 'stakehold', 'report', 'analysi']

Il tuo Syllabus offre queste parole:
hacking, offensive, exploitation, footprinting, scanning, enumeration.

Per un umano, Penetration Testing ~ Ethical Hacking.

Per l'algoritmo, la stringa "penetrat" è diversa dalla stringa "hack". 
Non si toccano.
L'unica parola che si incrocia è "analysis" (presente in "malware analysis" nel syllabus e "analysis" nel ruolo ENISA), che però è una parola generica e spesso ha un peso (KIF) basso perché sta in fondo alla lista delle keyword.